Parallel workfow with reducer function

In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
from langchain_core.messages import SystemMessage, HumanMessage
import operator

d:\FullStack_Practice\fullstack_practice\LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
class EvaluationSchema(BaseModel):
    feedback: str=Field(description="Feedback on the essay")
    score: int = Field(description="Score for the essay on a scale of 0-10", ge=0, le=10)

In [3]:
parser = PydanticOutputParser(pydantic_object=EvaluationSchema)

In [4]:
llm1 = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",  # provider-backed
    task="text-generation",
    temperature=0.1,
)

llm2=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    temperature=0.2,
)

llm3=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-Coder-Next",
    task="text-generation",
    temperature=0.2,
)


LamaChat = ChatHuggingFace(llm=llm1)
QwenChat=ChatHuggingFace(llm=llm2)
QwenCoderChat=ChatHuggingFace(llm=llm3)

In [5]:
essay = """Mental health is a fundamental aspect of overall well-being that deserves equal attention and care as physical health. It encompasses our emotional, psychological, and social well-being, influencing how we think, feel, and act in our daily lives. In today's fast-paced world, understanding and prioritizing mental health has become more important than ever.

Mental health affects every dimension of our lives. It impacts our relationships, work performance, and ability to handle life's challenges. When we maintain good mental health, we feel more confident, capable, and connected to those around us. Conversely, poor mental health can lead to anxiety, depression, and other conditions that significantly diminish quality of life.

Society has long stigmatized mental health issues, often treating them as weaknesses rather than legitimate health concerns. This stigma prevents many people from seeking help when they need it most. We must actively work to break down these barriers by promoting open conversations about mental health and creating supportive communities where people feel safe discussing their struggles.

Various factors contribute to mental health, including genetics, life experiences, and environmental conditions. Trauma, abuse, and chronic stress can negatively impact mental well-being. However, protective factors such as strong relationships, meaningful work, and access to resources can promote resilience and recovery. Understanding these factors helps us develop comprehensive approaches to mental health care.

Self-care plays a vital role in maintaining mental health. This includes regular exercise, adequate sleep, healthy nutrition, and engaging in activities that bring joy and relaxation. Mindfulness and meditation can help manage stress and anxiety, while journaling provides an outlet for processing emotions. Additionally, maintaining social connections and spending time with loved ones strengthens emotional support systems.

Professional help is essential when mental health challenges become overwhelming. Therapy, counseling, and psychiatric treatment provide evidence-based interventions that can transform lives. Mental health professionals help individuals develop coping strategies, address underlying issues, and build pathways toward healing and growth. Seeking professional support is a sign of strength, not weakness.

Workplace mental health is increasingly recognized as crucial for both employee well-being and organizational success. Employers who prioritize mental health through supportive policies, access to counseling services, and stress management programs create healthier, more productive workplaces. Companies that invest in their employees' mental health see improved retention and performance.

Schools and educational institutions must also embrace mental health as a priority. Young people face unprecedented pressures from academics, social media, and societal expectations. Providing mental health education, counseling services, and peer support programs empowers students to develop healthy coping mechanisms early in life.

Technology offers new opportunities for mental health support through teletherapy, mental health apps, and online communities. These tools make healthcare more accessible, particularly for those in remote areas or with mobility challenges. However, they must complement, not replace, in-person professional care.

Mental health advocacy at policy levels is crucial. Adequate funding for mental health services, insurance coverage for treatment, and legislation protecting mental health rights are essential changes. Governments and institutions must recognize mental health as a public health priority.

In conclusion, mental health is not a luxury but a necessity for living fulfilling lives. By promoting awareness, reducing stigma, providing resources, and supporting one another, we create societies where everyone can thrive. Prioritizing mental health is an investment in our collective future and well-being."""

In [6]:
class EssayState(TypedDict):
    essay_text: str
    cot_feedback: str # clarity of thought feedback
    doa_feedback: str # depth of analysis feedback
    language_feedback: str # language feedback
    individual_score: Annotated[list[int],operator.add] # score for the current essay
    avg_score: float # cumulative score for all essays so far
    overall_feedback: str # final feedback for the essay


In [18]:
def evaluate_language(state:EssayState):
    # evaluate language and update state
    prompt=PromptTemplate(template="""You are an essay evaluator, you will evaluate the essay on the parameter called language. You MUST return ONLY valid JSON.
Essay: {essay}
Return EXACTLY this format:
{format_instructions}
NO OTHER TEXT. ONLY JSON.""",
            input_variables=["essay"],
            partial_variables={"format_instructions": parser.get_format_instructions()}
)
    response=QwenChat.invoke(prompt.format(essay=state["essay_text"]))
    parsed_output = parser.parse(response.content) 
    return {"language_feedback":parsed_output.feedback,"individual_score":[parsed_output.score]}

In [19]:
def evaluate_clarity_of_thought(state:EssayState):
    prompt=PromptTemplate(template="""You are an essay evaluator, you will evaluate the essay on the parameter clarity of thought. You MUST return ONLY valid JSON.
Essay: {essay}
Return EXACTLY this format:
{format_instructions}
NO OTHER TEXT. ONLY JSON.""",
            input_variables=["essay"],
            partial_variables={"format_instructions": parser.get_format_instructions()}
)
    response=QwenCoderChat.invoke(prompt.format(essay=state["essay_text"]))
    parsed_output = parser.parse(response.content)
    return {"cot_feedback":parsed_output.feedback,"individual_score":[parsed_output.score]}

In [20]:
def evaluate_depth_of_analysis(state:EssayState):
    prompt=PromptTemplate(template="""You are an essay evaluator, you will evaluate the essay on the parameter depth of analysis. You MUST return ONLY valid JSON.
Essay: {essay}
Return EXACTLY this format:
{format_instructions}
NO OTHER TEXT. ONLY JSON.""",
            input_variables=["essay"],
            partial_variables={"format_instructions": parser.get_format_instructions()}
)
    response=LamaChat.invoke(prompt.format(essay=state["essay_text"]))
    parsed_output = parser.parse(response.content)
    return {"doa_feedback":parsed_output.feedback,"individual_score":[parsed_output.score]}

In [21]:
def final_evaluation(state:EssayState):

    # summary feedback
    prompt=PromptTemplate(template=""" You are an Feedback summarization specialist, you will create a summarized feedback based on the following feedbacks:
    \n\nlanguage feedback:{language_feedback}
    \n\ndepth of analysis feedback:{doa_feedback}
    \n\n clarity of thought feedback:{cot_feedback}""",
    input_variables=["language_feedback","doa_feedback","cot_feedback"]
    )
    response=LamaChat.invoke(prompt.format(language_feedback=state["language_feedback"],doa_feedback=state["doa_feedback"],cot_feedback=state["cot_feedback"]))
    #avg_score
    avg_score=sum(state["individual_score"])/len(state["individual_score"])

    return {"overall_feedback":response.content,"avg_score":avg_score}

In [22]:
graph=StateGraph(EssayState)

# define nodes
graph.add_node("evalute_langauge",evaluate_language)
graph.add_node("evalute_clarity_of_thought",evaluate_clarity_of_thought)
graph.add_node("evalute_depth_of_analysis",evaluate_depth_of_analysis)
graph.add_node("final_evaluation",final_evaluation)

# define edges
graph.add_edge(START,"evalute_langauge")
graph.add_edge(START,"evalute_clarity_of_thought")
graph.add_edge(START,"evalute_depth_of_analysis")
graph.add_edge("evalute_langauge","final_evaluation")
graph.add_edge("evalute_depth_of_analysis","final_evaluation")
graph.add_edge("evalute_clarity_of_thought","final_evaluation")

graph.add_edge("final_evaluation",END)

# compile graph
workflow=graph.compile()


In [23]:
initial_state = {"essay_text": essay}

workflow.invoke(initial_state)

{'essay_text': "Mental health is a fundamental aspect of overall well-being that deserves equal attention and care as physical health. It encompasses our emotional, psychological, and social well-being, influencing how we think, feel, and act in our daily lives. In today's fast-paced world, understanding and prioritizing mental health has become more important than ever.\n\nMental health affects every dimension of our lives. It impacts our relationships, work performance, and ability to handle life's challenges. When we maintain good mental health, we feel more confident, capable, and connected to those around us. Conversely, poor mental health can lead to anxiety, depression, and other conditions that significantly diminish quality of life.\n\nSociety has long stigmatized mental health issues, often treating them as weaknesses rather than legitimate health concerns. This stigma prevents many people from seeking help when they need it most. We must actively work to break down these bar

When workign with parallel workflow we can't return whole sate form indivisual node as langrph will assume updaing all valirble in that state of rhte node execution which will lead to confusion and failure.